# 📦 05 - Error Analysis, Pipeline Packaging & Deployment

Hoàn thiện giai đoạn cuối cùng theo yêu cầu đề bài:
- **1. Đánh giá cuối cùng trên tập Test độc lập** bằng mô hình chiến thắng và ngưỡng tối ưu
- **2. Phân tích lỗi sai chuyên sâu (Error Analysis)**: Mổ xẻ False Positives và False Negatives
- **3. Đóng gói toàn bộ Pipeline (Persistence)** thành file .joblib duy nhất
- **4. Triển khai dịch vụ suy luận (SpamInferenceService)**: Phân loại email mới unseen thời gian thực


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import display
from src.config import NOTEBOOK_SAVED_MODELS_DIR, SAVED_MODELS_DIR, NOTEBOOK_METRICS_DIR, METRICS_DIR
from src.models.persistence import load_spam_pipeline
from src.deployment.predictor import SpamInferenceService

# 1. Nạp pipeline đã đóng gói sẵn
pipeline_path = NOTEBOOK_SAVED_MODELS_DIR / 'spam_pipeline.joblib'
if not pipeline_path.exists():
    pipeline_path = SAVED_MODELS_DIR / 'spam_pipeline.joblib'

bundle = load_spam_pipeline(pipeline_path)

if bundle.get('feature_builder') is None and (SAVED_MODELS_DIR / 'spam_pipeline.joblib').exists():
    prod_bundle = load_spam_pipeline(SAVED_MODELS_DIR / 'spam_pipeline.joblib')
    bundle['feature_builder'] = prod_bundle.get('feature_builder')
    bundle['text_processor'] = prod_bundle.get('text_processor')

model = bundle['model']
best_threshold = float(bundle['best_threshold'])
metrics_test = bundle.get('test_metrics', bundle.get('metrics_test', {}))

# Bảng thông tin tổng quan Pipeline đã nạp
df_pipeline_overview = pd.DataFrame([
    {"Thông số Pipeline": "Đường dẫn file Joblib", "Giá trị": str(pipeline_path)},
    {"Thông số Pipeline": "Mô hình vô địch", "Giá trị": type(model).__name__},
    {"Thông số Pipeline": "Ngưỡng quyết định tối ưu (tau)", "Giá trị": f"{best_threshold:.4f}"},
])
display(df_pipeline_overview)

# Bảng kết quả đánh giá trên tập Test độc lập dạng DataFrame
if metrics_test:
    df_test_eval = pd.DataFrame([{
        "Tập dữ liệu": "Test Set độc lập (Unseen)",
        "Accuracy": round(metrics_test.get('accuracy', 0), 4),
        "Precision": round(metrics_test.get('precision', 0), 4),
        "Recall": f"{metrics_test.get('recall', 0):.4f} (Đạt >= 0.85: {metrics_test.get('recall', 0) >= 0.85})",
        "F1-Score": round(metrics_test.get('f1', 0), 4),
        "F-Beta (Beta=2.0)": round(metrics_test.get('f_beta', 0), 4),
        "ROC-AUC": round(metrics_test.get('roc_auc', 0), 4) if metrics_test.get('roc_auc') is not None else "N/A",
    }])
    display(df_test_eval)


,Thông số Pipeline,Giá trị
0,Đường dẫn file Joblib,E:\Hoc_May_Tren_truong\lab_1 spam_classifier\n...
1,Mô hình vô địch,SpamVotingEnsemble
2,Ngưỡng quyết định tối ưu (tau),0.3800


,Tập dữ liệu,Accuracy,Precision,Recall,F1-Score,F-Beta (Beta=2.0),ROC-AUC
0,Test Set độc lập (Unseen),0.6675,0.573,0.8676 (Đạt >= 0.85: True),0.6902,0.7867,N/A


## 1. Đánh giá cuối cùng trên tập Test độc lập


In [2]:
# Mổ xẻ phân tích lỗi sai từ file misclassified_samples.csv
csv_path = NOTEBOOK_METRICS_DIR / 'misclassified_samples.csv'
if not csv_path.exists():
    csv_path = METRICS_DIR / 'misclassified_samples.csv'

if csv_path.exists():
    error_df = pd.read_csv(csv_path)

    # 1. Bảng phân bố loại lỗi sai
    err_counts = error_df['error_type'].value_counts()
    df_err_dist = pd.DataFrame({
        "Loại lỗi sai (Error Category)": err_counts.index,
        "Số lượng mẫu": [f"{v:,}" for v in err_counts.values],
        "Tỷ lệ (%)": [f"{(v / len(error_df) * 100):.2f}%" for v in err_counts.values]
    })
    display(df_err_dist)

    # 2. Bảng TOP mẫu False Positives (Ham bị chặn nhầm - rủi ro cao)
    fp_samples = error_df[error_df['error_type'].str.contains('False Positive')].head(3)
    df_fp_table = pd.DataFrame([
        {
            "Mẫu": f"Email #{idx+1}",
            "Nội dung văn bản": str(row.get('text', ''))[:80] + ("..." if len(str(row.get('text', ''))) > 80 else ""),
            "Loại lỗi": "False Positive (Chặn nhầm)",
            "Lý do nghi ngờ": str(row.get('suspected_cause', 'N/A')),
        }
        for idx, row in fp_samples.iterrows()
    ])
    display(df_fp_table)

    # 3. Bảng TOP mẫu False Negatives (Spam lọt lưới)
    fn_samples = error_df[error_df['error_type'].str.contains('False Negative')].head(3)
    df_fn_table = pd.DataFrame([
        {
            "Mẫu": f"Email #{idx+1}",
            "Nội dung văn bản": str(row.get('text', ''))[:80] + ("..." if len(str(row.get('text', ''))) > 80 else ""),
            "Loại lỗi": "False Negative (Lọt lưới)",
            "Lý do nghi ngờ": str(row.get('suspected_cause', 'N/A')),
        }
        for idx, row in fn_samples.iterrows()
    ])
    display(df_fn_table)
else:
    print('Chưa tìm thấy file phân tích lỗi misclassified_samples.csv.')


,Loại lỗi sai (Error Category),Số lượng mẫu,Tỷ lệ (%)
0,False Positive (Chặn nhầm),"1,172",95.52%
1,False Negative (Lọt lưới),55,4.48%


,Mẫu,Nội dung văn bản,Loại lỗi,Lý do nghi ngờ
0,Email #1,"louise , i worked with the team this morning a...",False Positive (Chặn nhầm),Dính từ khóa nhạy cảm (call)
1,Email #2,. us email international business charset = is...,False Positive (Chặn nhầm),Dính từ khóa nhạy cảm (offer); Dấu chấm than (4)
2,Email #3,claim a free escapenumber victoria's secret gi...,False Positive (Chặn nhầm),"Dính từ khóa nhạy cảm (free, offer, claim)"


,Mẫu,Nội dung văn bản,Loại lỗi,Lý do nghi ngờ
0,Email #1173,betotgim di lehna,False Negative (Lọt lưới),"Email quá ngắn, thiếu từ khóa; Spam tinh vi, k..."
1,Email #1174,"hey , man ! : )\r\nca ndyi\r\n",False Negative (Lọt lưới),"Spam tinh vi, không dùng từ khóa lộ liễu"
2,Email #1175,tr tr tr tr fcajqzmysvgjcy negv h,False Negative (Lọt lưới),"Spam tinh vi, không dùng từ khóa lộ liễu"


## 2. Phân tích lỗi sai chuyên sâu (False Positive & False Negative)


In [3]:
# Bảng hiển thị chi tiết các thành phần trong Pipeline Bundle
df_bundle_content = pd.DataFrame([
    {
        "Thành phần trong Bundle (Key)": k,
        "Kiểu dữ liệu (Type)": type(bundle[k]).__name__,
        "Mô tả / Giá trị": str(bundle[k])[:65] + "..." if len(str(bundle[k])) > 65 else str(bundle[k]),
    }
    for k in bundle.keys()
])
display(df_bundle_content)


,Thành phần trong Bundle (Key),Kiểu dữ liệu (Type),Mô tả / Giá trị
0,model,SpamVotingEnsemble,<src.models.ensemble.SpamVotingEnsemble object...
1,model_name,str,SpamVotingEnsemble_Tuned
2,text_processor,SpamTextProcessor,<src.data.preprocessing.SpamTextProcessor obje...
3,feature_builder,HybridFeatureBuilderScratch,<src.features.feature_engineering.HybridFeatur...
4,best_threshold,float,0.38
5,validation_metrics,dict,"{'threshold': 0.38, 'tp': 1058, 'fp': 834, 'tn..."
6,test_metrics,dict,"{'threshold': 0.38, 'tp': 1075, 'fp': 801, 'tn..."


## 3. Đóng gói toàn bộ Pipeline Artifacts (.joblib)


In [4]:
service = SpamInferenceService(pipeline_bundle=bundle).load_service()

demo_emails = [
    'URGENT! You have won a 1,000,000 cash prize! Call 0900123456 now to claim your award immediately!',
    'Dear team, please find attached the weekly project status report and meeting notes for Monday.',
    'Free entry in 2 a weekly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question!',
    'Hey, are we still meeting for lunch at 12:30 today? Let me know!',
]

demo_results = service.predict_batch(demo_emails)
display(demo_results[['prediction', 'is_spam', 'spam_probability', 'risk_level', 'detected_signals']])


,prediction,is_spam,spam_probability,risk_level,detected_signals
0,SPAM,True,0.7873,Cao,"[Từ khóa nhạy cảm: call, cash, claim, prize, u..."
1,NOT SPAM / HAM,False,0.2814,Trung bình,[]
2,SPAM,True,0.7061,Cao,"[Từ khóa nhạy cảm: free, win]"
3,NOT SPAM / HAM,False,0.3413,Trung bình,[]


## 4. Triển khai dịch vụ suy luận & Demo dự đoán email mới (Deployment)


In [5]:
service = SpamInferenceService(pipeline_bundle=bundle).load_service()

demo_emails = [
    'URGENT! You have won a 1,000,000 cash prize! Call 0900123456 now to claim your award immediately!',
    'Dear team, please find attached the weekly project status report and meeting notes for Monday.',
    'Free entry in 2 a weekly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question!',
    'Hey, are we still meeting for lunch at 12:30 today? Let me know!',
]

demo_results = service.predict_batch(demo_emails)
demo_results[['prediction', 'is_spam', 'spam_probability', 'risk_level', 'detected_signals']]


,prediction,is_spam,spam_probability,risk_level,detected_signals
0,SPAM,True,0.7873,Cao,"[Từ khóa nhạy cảm: call, cash, claim, prize, u..."
1,NOT SPAM / HAM,False,0.2814,Trung bình,[]
2,SPAM,True,0.7061,Cao,"[Từ khóa nhạy cảm: free, win]"
3,NOT SPAM / HAM,False,0.3413,Trung bình,[]
